# 02 - Train Battery RUL PINN-LSTM

This notebook trains the Smart TwinPac battery Remaining Useful Life model.

**Input files**

- `data/battery_train.csv`
- `data/battery_test.csv`
- `data/battery_stats.json`

**Expected outputs**

- `models/battery_rul_pinn_lstm.h5`
- `models/battery_rul_model_info.json`

**Targets**

- Test MAE < 30 days
- R2 > 0.90

## Mathematical foundation

The physics branch is inspired by the Shepherd discharge model:

`V(t) = E0 - R*i(t) - K*(Q/(Q-q(t)))*(i(t)+q(t)) + A*exp(-B*q(t))`

For the prototype, the branch learns a voltage residual proxy from the last state of the 180-day window. This keeps training simple while still penalizing predictions that ignore discharge voltage behavior.

In [ ]:
# Colab setup if needed:
# !pip install tensorflow pandas numpy

from backend.ml.battery_rul_pinn_lstm import (
    TrainingConfig,
    load_battery_data,
    create_sequences,
    build_pinn_lstm,
    train_battery_rul_model,
)

config = TrainingConfig(
    train_csv='data/battery_train.csv',
    test_csv='data/battery_test.csv',
    stats_json='data/battery_stats.json',
    model_output='models/battery_rul_pinn_lstm.h5',
    info_output='models/battery_rul_model_info.json',
    window_days=180,
    batch_size=16,
    epochs=100,
    alpha_physics=0.3,
    beta_prediction=0.7,
)
config

## Load cleaned data and build windows

The cleaned dataset is monthly. The training module interpolates each sequence to daily resolution, then creates 180-day windows with input shape `(batch, 180, 3)`.

In [ ]:
train_df, test_df, stats = load_battery_data(config)
X_train, y_train, y_train_voltage = create_sequences(train_df, stats, window_days=config.window_days)
X_test, y_test, y_test_voltage = create_sequences(test_df, stats, window_days=config.window_days)

print(f'Training shape: X={X_train.shape}, y={y_train.shape}')
print(f'Test shape: X={X_test.shape}, y={y_test.shape}')

## Build PINN-LSTM architecture

- Physics branch: last state -> Dense(64, tanh) -> Dense(32, tanh) -> voltage residual output
- LSTM branch: LSTM(128) -> LSTM(64) -> Dense(32)
- Fusion: concatenate physics and temporal features -> Dense(16) -> RUL days
- Total loss: `0.3 * physics_voltage_mse + 0.7 * rul_mae`

In [ ]:
model = build_pinn_lstm(input_shape=(180, 3), alpha=0.3, beta=0.7)
model.summary()

## Train and export

This cell trains the model and saves both `.h5` weights and metadata JSON. In Colab free tier, start with fewer epochs for a smoke test, then increase to 100.

In [ ]:
model_info = train_battery_rul_model(config)
model_info

## Metrics to report

Report:

- Test MAE in days
- R2 score
- Physics voltage MSE
- alpha/beta loss weights
- Number of train/test windows
- Whether target MAE < 30 days is met

In [ ]:
print('=' * 60)
print('BATTERY RUL MODEL - FINAL METRICS')
print('=' * 60)
print(f"Test MAE: {model_info['mae_days']:.2f} days (Target: <30 days)")
print(f"Test R2: {model_info['r2_score']:.4f} (Target: >0.90)")
print(f"Physics voltage MSE: {model_info['physics_voltage_mse']:.6f}")
print(f"Target met: {model_info['target_met']}")
print('=' * 60)

## Hyperparameter tuning if MAE > 30 days

1. Increase window: `180 -> 210` days.
2. Adjust loss weights: `alpha=0.2`, `beta=0.8` for more data-driven behavior.
3. Add BatchNormalization after LSTM layers.
4. Increase LSTM units: `128 -> 256`, `64 -> 128`.

The model metadata JSON records the current configuration so experiments remain traceable.